In [50]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

import pickle

In [21]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout
)

In [27]:
df=pd.read_csv(
    "../datasets/metadata_classifier_dataset.csv"
)

df.head()

,metadata,label
0,Nike Air Force 1 Low Brown Grey Classic Lifest...,0
1,Nike Air Force 1 Low Brown Grey Classic Lifest...,1
2,Nike Air Max Grey Black Classic Lifestyle Snea...,0
3,Nike Air Max Grey Black Classic Lifestyle Snea...,1
4,Nike Air Force 1 Low Grey Charcoal Low Basketb...,0


In [28]:
X_train, X_test, y_train, y_test=train_test_split(
    df["metadata"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [29]:
MAX_WORDS=10000
MAX_LEN=110

In [30]:
tokenizer=Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(
    X_train
)

In [31]:
l=[len(x) for x in X_train]
print(np.max(l))
print(np.mean(l))

113
65.91175774781169


In [32]:
X_train_seq = tokenizer.texts_to_sequences(
    X_train
)

X_test_seq = tokenizer.texts_to_sequences(
    X_test
)

In [33]:
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding="post"
)

In [34]:
model = Sequential([
    Input(shape=(MAX_LEN,)),

    Embedding(
        input_dim=MAX_WORDS,   # vocabulary size
        output_dim=128         # embedding dimension
    ),

    Bidirectional(
        LSTM(
            128,
            return_sequences=False
        )
    ),

    Dropout(0.3),

    Dense(
        64,
        activation="relu"
    ),

    Dropout(0.3),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [35]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [36]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)              │ (None, 110, 128)            │       1,280,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_2 (Bidirectional)      │ (None, 256)                 │         263,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 64)                  │          16,448 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,559,681 (5.95 MB)

 Trainable params: 1,559,681 (5.95 MB)

 Non-trainable params: 0 (0.00 B)

In [37]:
history = model.fit(
    X_train_pad,
    y_train,

    validation_data=(X_test_pad, y_test),

    epochs=10,

    batch_size=32
)

Epoch 1/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 11s 63ms/step - accuracy: 0.9238 - loss: 0.1654 - val_accuracy: 0.9972 - val_loss: 0.0052
Epoch 2/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 8s 59ms/step - accuracy: 0.9998 - loss: 0.0017 - val_accuracy: 1.0000 - val_loss: 0.0014
Epoch 3/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 8s 59ms/step - accuracy: 1.0000 - loss: 1.8548e-04 - val_accuracy: 1.0000 - val_loss: 1.0334e-04
Epoch 4/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 8s 60ms/step - accuracy: 1.0000 - loss: 9.4879e-05 - val_accuracy: 1.0000 - val_loss: 1.1394e-04
Epoch 5/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 8s 60ms/step - accuracy: 1.0000 - loss: 3.1664e-05 - val_accuracy: 1.0000 - val_loss: 7.6291e-05
Epoch 6/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 8s 61ms/step - accuracy: 1.0000 - loss: 2.5063e-05 - val_accuracy: 1.0000 - val_loss: 7.3689e-05
Epoch 7/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 8s 61ms/step - accuracy: 1.0000 - loss: 3.8484e-05 - val_accuracy: 1.0000 - val_loss: 3.5585e-05
Epoch 8/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 8s 60ms/step - a

In [38]:
loss, acc=model.evaluate(
    X_test_pad,
    y_test
)

print(
    f"Test Accuracy: {acc:.4f}"
)

34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 1.0000 - loss: 3.4844e-06
Test Accuracy: 1.0000


In [40]:
pred_probs=model.predict(
    X_test_pad
)

preds=(
    pred_probs > 0.5
).astype(int)

print(
    classification_report(
        y_test,
        preds
    )
)

34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       529
           1       1.00      1.00      1.00       528

    accuracy                           1.00      1057
   macro avg       1.00      1.00      1.00      1057
weighted avg       1.00      1.00      1.00      1057



In [49]:
model.save(
    "metadata_counterfeit_classifier_v1.keras"
)

In [51]:
with open(
    "metadata_tokenizer.pkl",
    "wb"
) as f:

    pickle.dump(
        tokenizer,
        f
    )

In [52]:
samples = [

    # Genuine
    "Nike Air Zoom Pegasus 40 Grey White Running Shoes",
    "Puma RS-X White Navy Lifestyle Sneakers",
    "New Balance 574 Grey Navy Casual Trainers",
    "Vans Sk8-Hi Black White Skate Shoes",
    "Reebok Club C 85 White Green Lifestyle Sneakers",
    "Under Armour Charged Pursuit Grey Black Running Shoes",
    "Nike Air Max 90 White Red Casual Sneakers",
    "New Balance 550 White Grey Retro Basketball Sneakers",

    # Hard Counterfeit
    "Nike Air Zoom Pegasus 40 Grey White Running Shoes Retail Version",
    "Puma RS-X White Navy Lifestyle Sneakers Store Selection",
    "New Balance 574 Grey Navy Casual Trainers Premium Release",
    "Vans Sk8-Hi Black White Skate Shoes Executive Collection",
    "Reebok Club C 85 White Green Lifestyle Sneakers Fashion Edition",
    "Under Armour Charged Pursuit Grey Black Running Shoes Imported Version",
    "Nike Air Max 90 White Red Casual Sneakers Urban Selection",
    "New Balance 550 White Grey Retro Basketball Sneakers Special Release",

    # Extremely Hard Counterfeit
    "Nike Air Max 90 White Red Lifestyle Sneakers Exclusive Series",
    "Puma RS-X Grey White Lifestyle Sneakers Signature Edition",
    "New Balance 574 Navy Grey Casual Trainers Heritage Collection",
    "Vans Old Skool Black White Skate Shoes Limited Selection",
    "Reebok Club C White Green Lifestyle Sneakers Premium Series",
    "Under Armour Charged Pursuit Black Grey Running Shoes Modern Edition",

    # Brand Consistency Stress Test
    "Puma Air Force 1 White Black Casual Sneakers",
    "Vans Air Max 90 White Grey Running Shoes",
    "New Balance Sk8-Hi Black White Skate Shoes",
    "Reebok RS-X White Navy Lifestyle Sneakers",
    "Nike Club C 85 White Green Casual Sneakers"

]

sample_sequences = tokenizer.texts_to_sequences(samples)

sample_padded = pad_sequences(
    sample_sequences,
    maxlen=MAX_LEN,
    padding='post'
)

test_pred_probs = model.predict(sample_padded)

test_preds = (test_pred_probs > 0.5).astype(int)

for text, prob, pred in zip(samples, test_pred_probs, test_preds):
    print(f"{pred[0]} | {prob[0]:.4f} | {text}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
0 | 0.0074 | Nike Air Zoom Pegasus 40 Grey White Running Shoes
0 | 0.0000 | Puma RS-X White Navy Lifestyle Sneakers
0 | 0.0000 | New Balance 574 Grey Navy Casual Trainers
0 | 0.0000 | Vans Sk8-Hi Black White Skate Shoes
0 | 0.0000 | Reebok Club C 85 White Green Lifestyle Sneakers
0 | 0.0000 | Under Armour Charged Pursuit Grey Black Running Shoes
0 | 0.0000 | Nike Air Max 90 White Red Casual Sneakers
0 | 0.0000 | New Balance 550 White Grey Retro Basketball Sneakers
1 | 1.0000 | Nike Air Zoom Pegasus 40 Grey White Running Shoes Retail Version
1 | 1.0000 | Puma RS-X White Navy Lifestyle Sneakers Store Selection
1 | 1.0000 | New Balance 574 Grey Navy Casual Trainers Premium Release
1 | 1.0000 | Vans Sk8-Hi Black White Skate Shoes Executive Collection
1 | 1.0000 | Reebok Club C 85 White Green Lifestyle Sneakers Fashion Edition
1 | 1.0000 | Under Armour Charged Pursuit Grey Black Running Shoes Imported Version
1 | 1.0000 | Nike Air Max 90 White Red Casua